# Configured multilevel MOT trajectory lab

Use this notebook to define a conventional-MOT configuration, inspect its optical and magnetic geometry, and run one or more deterministic trajectories with the replacement Section-12 24-state population-rate solver.

The solver recomputes the local beam intensities, magnetic field, Doppler and Zeeman shifts, polarization decomposition, steady-state populations, scattering, and force at every RK4 stage. Recoil diffusion is not implemented, so these are deterministic mean-force trajectories. Ensemble members differ only through their seeded initial geometry (and optional seeded launch-speed spread). Results remain diagnostic until the multilevel capture/loading validation program is complete.

## 1. Imports and model initialization

The ARC-derived 24-state atomic structure is built once and cached. This cell can take a little time on the first run.

In [ ]:
%matplotlib inline
from dataclasses import asdict, replace
from datetime import datetime
import json
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from pmot.beam_plotting import beam_surface_mesh_mm
from pmot.configuration import default_mot_apparatus_config
from pmot.launch_geometry import build_incident_disc_from_angles
from pmot.magnetic_fields import anti_helmholtz_field_t, default_anti_helmholtz_config
from pmot.mot_multilevel import (
    CaptureSearchConfig,
    RateEquationAtomState,
    RateEquationTrajectoryConfig,
    build_multilevel_mot_beams,
    build_rate_equation_model,
    create_trajectory_animation,
    default_multilevel_mot_config,
    draw_mot_beam_volumes,
    generate_capture_launches,
    multilevel_mot_paths,
    plot_time_diagnostics,
    plot_trajectory_3d,
    save_trajectory,
    simulate_rate_equation_trajectory,
    trajectory_summary,
)

MODEL = build_rate_equation_model()
OUTPUT_PATHS = multilevel_mot_paths()
print(f'Loaded {MODEL.state_count} states and {MODEL.transition_count} allowed transitions.')

## 2. Physical MOT configuration

Edit the values below, then rerun this cell and the **Build and review configuration** cell. Powers are per traveling beam. Detunings use ordinary frequency in MHz here and are converted to angular frequency internally. Beam diameters are Gaussian $1/e^2$ intensity diameters.

The replacement builder currently uses the canonical propagation-frame helicities `(x, y, z) = (sigma+, sigma+, sigma-)` for both directions on each Cartesian path. Gravity can be disabled for force-symmetry diagnostics.

In [ ]:
PHYSICS = {
    # Cooling and repump light
    'cooling_power_mw_per_beam': 27.0,
    'repump_power_mw_per_beam': 0.1,
    'cooling_detuning_mhz': -15.0,
    'repump_detuning_mhz': 0.0,
    'cooling_wavelength_nm': 780.0,
    'repump_wavelength_nm': 780.232684,
    'cooling_beam_diameter_mm': 12.7,
    'repump_beam_diameter_mm': 12.7,
    'repumper_enabled': True,

    # Anti-Helmholtz pair. The current is derived from the requested central axial gradient.
    'axial_gradient_g_per_cm': 10.0,
    'coil_radius_mm': 40.0,
    'turns_per_coil': 50,

    # External dynamics
    'include_gravity': True,
}

pd.Series(PHYSICS, name='value').to_frame()

## 3. Numerical and launch configuration

Choose `launch_mode = 'manual'` for one explicit Cartesian initial condition, `launch_mode = 'specified_disc'` for an exact direction-disc launch, or `launch_mode = 'random_full_sphere'` for a seeded ensemble.

For `specified_disc`, the launch polar and azimuthal angles locate the center of the sampling disc on the launch sphere; the incoming velocity is antiparallel to that outward direction. The exact impact parameter fixes the transverse distance from the centered ray. `impact_azimuth_deg` fixes its orientation within the disc plane, which is also required to identify a unique starting point.

A randomized launch samples an incident direction uniformly over the full sphere. Its starting point is sampled uniformly in area on a disc perpendicular to that direction, and its velocity is parallel to the disc normal toward the trap. `trajectory_count` is the number of independent direction discs, with one launch point per disc. Set `impact_disc_radius_mm = 0` for centered radial launches.

In [ ]:
SIMULATION = {
    # Integration
    'duration_ms': 5.0,
    'time_step_us': 5.0,
    'escape_radius_mm': 30.0,

    # Select 'manual', 'specified_disc', or 'random_full_sphere'
    'launch_mode': 'random_full_sphere',
    'trajectory_count': 1,
    'random_seed': 20261006,

    # Random full-sphere geometry
    'launch_radius_mm': 15.0,
    'impact_disc_radius_mm': 5.0,
    'launch_speed_m_per_s': 3.0,
    'launch_speed_std_m_per_s': 0.0,

    # Exact direction-disc geometry used only by launch_mode='specified_disc'.
    # Polar angle is measured from +z; azimuth is measured from +x toward +y.
    'launch_polar_angle_deg': 90.0,
    'launch_azimuthal_angle_deg': 0.0,
    'impact_parameter_mm': 0.0,
    'impact_azimuth_deg': 0.0,

    # Manual Cartesian initial condition
    'manual_position_mm': (8.0, 1.0, 0.0),
    'manual_velocity_m_per_s': (-3.0, 0.1, 0.0),

    # Visualization and persistence
    'field_plot_extent_mm': 15.0,
    'field_plot_samples_per_axis': 81,
    'selected_trajectory_index': 0,
    'animation_max_frames': 160,  # evenly spaced across the full trajectory
    'animation_fps': 20,
    'save_results': False,
    'save_animation_gif': False,
    'run_name': 'configured_trajectory_lab',
}

pd.Series(SIMULATION, name='value').to_frame()

## 4. Build and review the configuration

This cell validates units and ranges, constructs the apparatus, optical beams, coil pair, numerical controls, and initial states, and prints the exact realized configuration. Run it again whenever either parameter dictionary changes.

In [ ]:
def _require_positive(mapping, keys):
    for key in keys:
        if float(mapping[key]) <= 0.0:
            raise ValueError(f'{key} must be positive')


def build_notebook_configuration(physics, simulation):
    _require_positive(physics, [
        'cooling_beam_diameter_mm', 'repump_beam_diameter_mm',
        'cooling_wavelength_nm', 'repump_wavelength_nm',
        'axial_gradient_g_per_cm', 'coil_radius_mm', 'turns_per_coil',
    ])
    _require_positive(simulation, [
        'duration_ms', 'time_step_us', 'escape_radius_mm',
        'launch_radius_mm', 'launch_speed_m_per_s',
        'field_plot_extent_mm', 'field_plot_samples_per_axis',
        'animation_max_frames', 'animation_fps',
    ])
    if physics['cooling_power_mw_per_beam'] < 0 or physics['repump_power_mw_per_beam'] < 0:
        raise ValueError('beam powers must be non-negative')
    if (
        simulation['impact_disc_radius_mm'] < 0
        or simulation['impact_parameter_mm'] < 0
        or simulation['launch_speed_std_m_per_s'] < 0
    ):
        raise ValueError('disc radius, impact parameter, and speed standard deviation must be non-negative')
    if simulation['launch_mode'] not in {'manual', 'specified_disc', 'random_full_sphere'}:
        raise ValueError("launch_mode must be 'manual', 'specified_disc', or 'random_full_sphere'")
    if not 0.0 <= float(simulation['launch_polar_angle_deg']) <= 180.0:
        raise ValueError('launch_polar_angle_deg must lie between 0 and 180 degrees')
    if int(simulation['trajectory_count']) < 1:
        raise ValueError('trajectory_count must be at least one')
    if int(simulation['field_plot_samples_per_axis']) < 11:
        raise ValueError('field_plot_samples_per_axis must be at least 11')

    rate_config = replace(
        default_multilevel_mot_config(),
        cooling_detuning_rad_per_s=2.0 * np.pi * physics['cooling_detuning_mhz'] * 1.0e6,
        repump_detuning_rad_per_s=2.0 * np.pi * physics['repump_detuning_mhz'] * 1.0e6,
        cooling_power_w_per_beam=physics['cooling_power_mw_per_beam'] * 1.0e-3,
        repump_power_w_per_beam=physics['repump_power_mw_per_beam'] * 1.0e-3,
        cooling_wavelength_m=physics['cooling_wavelength_nm'] * 1.0e-9,
        repump_wavelength_m=physics['repump_wavelength_nm'] * 1.0e-9,
        repumper_enabled=bool(physics['repumper_enabled']),
        include_gravity=bool(physics['include_gravity']),
    )

    base_apparatus = default_mot_apparatus_config()
    apparatus = replace(
        base_apparatus,
        cooling=replace(
            base_apparatus.cooling,
            beam_diameter_m=physics['cooling_beam_diameter_mm'] * 1.0e-3,
        ),
        repump=replace(
            base_apparatus.repump,
            beam_diameter_m=physics['repump_beam_diameter_mm'] * 1.0e-3,
        ),
    )
    coil = default_anti_helmholtz_config(
        radius_m=physics['coil_radius_mm'] * 1.0e-3,
        turns_per_coil=int(physics['turns_per_coil']),
        target_gradient_g_per_cm=physics['axial_gradient_g_per_cm'],
    )
    beams = build_multilevel_mot_beams(apparatus_config=apparatus, config=rate_config)
    numerical = RateEquationTrajectoryConfig(
        time_step_s=simulation['time_step_us'] * 1.0e-6,
        escape_radius_m=simulation['escape_radius_mm'] * 1.0e-3,
    )

    launch_metadata = []
    if simulation['launch_mode'] == 'manual':
        position = tuple(1.0e-3 * np.asarray(simulation['manual_position_mm'], dtype=float))
        velocity = tuple(np.asarray(simulation['manual_velocity_m_per_s'], dtype=float))
        initial_states = [RateEquationAtomState(position, velocity)]
        launch_metadata.append({'mode': 'manual', 'position_m': position, 'velocity_m_per_s': velocity})
    elif simulation['launch_mode'] == 'specified_disc':
        polar_rad = float(np.deg2rad(simulation['launch_polar_angle_deg']))
        azimuthal_rad = float(np.deg2rad(simulation['launch_azimuthal_angle_deg']))
        impact_azimuth_rad = float(np.deg2rad(simulation['impact_azimuth_deg']))
        impact_parameter_m = float(simulation['impact_parameter_mm']) * 1.0e-3
        disc = build_incident_disc_from_angles(
            0,
            simulation['launch_radius_mm'] * 1.0e-3,
            polar_rad,
            azimuthal_rad,
        )
        transverse_offset = impact_parameter_m * (
            np.cos(impact_azimuth_rad) * np.asarray(disc.basis_u, dtype=float)
            + np.sin(impact_azimuth_rad) * np.asarray(disc.basis_v, dtype=float)
        )
        position_array = np.asarray(disc.center_position_m, dtype=float) + transverse_offset
        velocity_array = (
            float(simulation['launch_speed_m_per_s'])
            * np.asarray(disc.incident_unit_vector, dtype=float)
        )
        position = tuple(float(value) for value in position_array)
        velocity = tuple(float(value) for value in velocity_array)
        initial_states = [RateEquationAtomState(position, velocity)]
        launch_metadata.append({
            'mode': 'specified_disc',
            'theta_rad': polar_rad,
            'phi_rad': azimuthal_rad,
            'impact_azimuth_rad': impact_azimuth_rad,
            'impact_radius_m': impact_parameter_m,
            'disc_center_position_m': disc.center_position_m,
            'position_m': position,
            'velocity_m_per_s': velocity,
        })
    else:
        count = int(simulation['trajectory_count'])
        # The geometry helper requires a positive disc radius; use an effectively
        # zero disc when the user requests centered radial launches.
        requested_disc_radius_m = simulation['impact_disc_radius_mm'] * 1.0e-3
        sampling_disc_radius_m = max(requested_disc_radius_m, np.finfo(float).eps)
        launch_search = CaptureSearchConfig(
            radial_distance_m=simulation['launch_radius_mm'] * 1.0e-3,
            disc_radius_m=sampling_disc_radius_m,
            disc_count=count,
            points_per_disc=1,
            include_center_point=(requested_disc_radius_m == 0.0),
            maximum_simulation_time_s=simulation['duration_ms'] * 1.0e-3,
            time_step_s=simulation['time_step_us'] * 1.0e-6,
            escape_radius_m=simulation['escape_radius_mm'] * 1.0e-3,
            seed=int(simulation['random_seed']),
        )
        _, points = generate_capture_launches(launch_search)
        speed_rng = np.random.default_rng(int(simulation['random_seed']) + 1)
        initial_states = []
        for point in points:
            speed = float(simulation['launch_speed_m_per_s'])
            if simulation['launch_speed_std_m_per_s'] > 0.0:
                speed = max(0.0, float(speed_rng.normal(speed, simulation['launch_speed_std_m_per_s'])))
            velocity = tuple(speed * np.asarray(point.incident_unit_vector, dtype=float))
            initial_states.append(RateEquationAtomState(point.initial_position_m, velocity))
            launch_metadata.append({
                'mode': 'random_full_sphere',
                'disc_index': point.disc_index,
                'theta_rad': point.theta_rad,
                'phi_rad': point.phi_rad,
                'impact_radius_m': point.s_m if requested_disc_radius_m > 0.0 else 0.0,
                'position_m': point.initial_position_m,
                'velocity_m_per_s': velocity,
            })

    return rate_config, apparatus, coil, beams, numerical, initial_states, launch_metadata


RATE_CONFIG, APPARATUS, COIL, BEAMS, NUMERICAL, INITIAL_STATES, LAUNCH_METADATA = (
    build_notebook_configuration(PHYSICS, SIMULATION)
)

beam_rows = [{
    'label': beam.label,
    'family': beam.family,
    'axis': beam.axis_name,
    'direction': beam.direction,
    'polarization': beam.circular_polarization,
    'diameter_mm': 2.0e3 * beam.beam_radius_m,
    'power_mW': 1.0e3 * beam.power_w,
    'detuning_MHz': beam.detuning_hz * 1.0e-6,
} for beam in BEAMS]
display(pd.DataFrame(beam_rows))
display(pd.DataFrame(LAUNCH_METADATA))
print(f'Coil current: {COIL.current_a:.6g} A')
print(f'Trajectories: {len(INITIAL_STATES)}')
print(f'Nominal RK4 steps per full-duration trajectory: {int(np.ceil(SIMULATION["duration_ms"] * 1000 / SIMULATION["time_step_us"])):,}')

## 5. Optical-beam geometry

The translucent cylinders show the true configured Gaussian $1/e^2$ diameters. Cooling and repump envelopes are drawn separately, so differing diameters remain visible. The thin Cartesian lines mark the coordinate axes and the trap center is at the origin.

In [ ]:
def plot_configured_beam_geometry(beams, length_mm=60.0):
    colors = {'cooling': '#ec4899', 'repump': '#2563eb'}
    alphas = {'cooling': 0.16, 'repump': 0.10}
    figure = plt.figure(figsize=(9.5, 8.0), constrained_layout=True)
    axis = figure.add_subplot(111, projection='3d')
    drawn = set()
    for beam in beams:
        key = (beam.family, beam.axis_name)
        if key in drawn:
            continue
        drawn.add(key)
        x, y, z = beam_surface_mesh_mm(
            beam.direction, beam.beam_radius_m, length_mm * 1.0e-3,
            axial_samples=18, angular_samples=32,
        )
        axis.plot_surface(
            x, y, z, color=colors.get(beam.family, '#64748b'),
            linewidth=0.0, shade=False, alpha=alphas.get(beam.family, 0.10),
        )
    half = 0.5 * length_mm
    axis.plot([-half, half], [0, 0], [0, 0], color='#b91c1c', linewidth=0.8)
    axis.plot([0, 0], [-half, half], [0, 0], color='#1d4ed8', linewidth=0.8)
    axis.plot([0, 0], [0, 0], [-half, half], color='#15803d', linewidth=0.8)
    axis.scatter(0, 0, 0, color='#111827', s=45)
    axis.set(
        xlim=(-half, half), ylim=(-half, half), zlim=(-half, half),
        xlabel='x [mm]', ylabel='y [mm]', zlabel='z [mm]',
        title='Configured MOT beam envelopes at true in-trap scale',
    )
    axis.set_box_aspect((1, 1, 1))
    families = sorted({beam.family for beam in beams})
    axis.legend(handles=[Patch(color=colors[name], alpha=0.35, label=name) for name in families])
    return figure

BEAM_GEOMETRY_FIGURE = plot_configured_beam_geometry(BEAMS)
plt.show()

## 6. Magnetic-field maps and central gradients

Each row shows a Cartesian plane. The left column is field magnitude and the right column is the in-plane vector field. The lineouts below show the diagonal field components along the three principal axes and report fitted central slopes in G/cm. These are field diagnostics, not optical-force plots.

In [ ]:
def plot_field_planes_and_gradients(coil, extent_mm, sample_count):
    coordinates_m = np.linspace(-extent_mm, extent_mm, int(sample_count)) * 1.0e-3
    first, second = np.meshgrid(coordinates_m, coordinates_m)
    plane_specs = {
        'xy': ((first, second, 0.0), (0, 1)),
        'xz': ((first, 0.0, second), (0, 2)),
        'yz': ((0.0, first, second), (1, 2)),
    }
    figure, panels = plt.subplots(3, 2, figsize=(13, 17), constrained_layout=True)
    extent = (-extent_mm, extent_mm, -extent_mm, extent_mm)
    stride = max(1, int(sample_count) // 15)
    coordinate_mm = 1.0e3 * coordinates_m
    grid_1_mm, grid_2_mm = np.meshgrid(coordinate_mm, coordinate_mm)
    for row, (plane, (arguments, components)) in enumerate(plane_specs.items()):
        field_g = [1.0e4 * np.asarray(value) for value in anti_helmholtz_field_t(*arguments, coil)]
        magnitude_g = np.sqrt(sum(component**2 for component in field_g))
        image = panels[row, 0].imshow(
            magnitude_g, origin='lower', extent=extent, aspect='equal', cmap='magma',
        )
        figure.colorbar(image, ax=panels[row, 0], label='|B| [G]')
        panels[row, 0].set_title(f'Field magnitude in {plane.upper()} plane')
        panels[row, 1].quiver(
            grid_1_mm[::stride, ::stride], grid_2_mm[::stride, ::stride],
            field_g[components[0]][::stride, ::stride],
            field_g[components[1]][::stride, ::stride],
            magnitude_g[::stride, ::stride], cmap='viridis', pivot='mid',
        )
        panels[row, 1].set_title(f'In-plane field vectors in {plane.upper()} plane')
        for column in range(2):
            panels[row, column].set_xlabel(f'{plane[0]} [mm]')
            panels[row, column].set_ylabel(f'{plane[1]} [mm]')
            panels[row, column].set_aspect('equal')
            panels[row, column].grid(alpha=0.16)

    line_figure, line_axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
    component_names = ('Bx', 'By', 'Bz')
    gradient_rows = []
    central = np.abs(coordinates_m) <= max(1.0e-3, 0.15 * extent_mm * 1.0e-3)
    for index, (axis_name, component_name) in enumerate(zip('xyz', component_names)):
        xyz = [np.zeros_like(coordinates_m) for _ in range(3)]
        xyz[index] = coordinates_m
        field_components_g = [1.0e4 * np.asarray(value) for value in anti_helmholtz_field_t(*xyz, coil)]
        values_g = field_components_g[index]
        slope_g_per_m = float(np.polyfit(coordinates_m[central], values_g[central], 1)[0])
        slope_g_per_cm = slope_g_per_m / 100.0
        gradient_rows.append({'axis': axis_name, 'component': component_name, 'central_slope_G_per_cm': slope_g_per_cm})
        line_axes[index].plot(coordinate_mm, values_g, color=('#b91c1c', '#1d4ed8', '#15803d')[index])
        line_axes[index].set(
            title=f'{component_name} along {axis_name}; slope={slope_g_per_cm:.4g} G/cm',
            xlabel=f'{axis_name} [mm]', ylabel=f'{component_name} [G]',
        )
        line_axes[index].axhline(0.0, color='#64748b', linewidth=0.7)
        line_axes[index].axvline(0.0, color='#64748b', linewidth=0.7)
        line_axes[index].grid(alpha=0.22)
    return figure, line_figure, pd.DataFrame(gradient_rows)

FIELD_PLANE_FIGURE, FIELD_LINEOUT_FIGURE, CENTRAL_GRADIENTS = plot_field_planes_and_gradients(
    COIL, SIMULATION['field_plot_extent_mm'], SIMULATION['field_plot_samples_per_axis'],
)
display(CENTRAL_GRADIENTS)
plt.show()

## 7. Run the trajectories

The full 24-state steady-state problem is solved four times per RK4 step, so runtime grows approximately with trajectory count, duration, and inverse timestep. Start with one short trajectory when testing a new configuration.

When `save_results` is enabled, each trajectory is stored as compressed NPZ, flat CSV, and self-describing metadata JSON under `outputs/trajectories/mot_multilevel_population_rate_v1/notebook_configured_trajectory_lab/<run name>/`. Figures go to the corresponding figures namespace.

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
safe_run_name = ''.join(character if character.isalnum() or character in '-_' else '_' for character in SIMULATION['run_name']).strip('_')
if not safe_run_name:
    raise ValueError('run_name must contain at least one letter or number')
trajectory_root = OUTPUT_PATHS['trajectories'] / 'notebook_configured_trajectory_lab' / f'{safe_run_name}_{timestamp}'
figure_root = OUTPUT_PATHS['figures'] / 'notebook_configured_trajectory_lab' / f'{safe_run_name}_{timestamp}'

RECORDS = []
SUMMARY_ROWS = []
ensemble_started = perf_counter()
for index, initial_state in enumerate(INITIAL_STATES):
    started = perf_counter()
    print(f'Running trajectory {index + 1}/{len(INITIAL_STATES)} ...', flush=True)
    record = simulate_rate_equation_trajectory(
        initial_state,
        SIMULATION['duration_ms'] * 1.0e-3,
        COIL,
        beams=BEAMS,
        model=MODEL,
        config=RATE_CONFIG,
        trajectory_config=NUMERICAL,
    )
    wall_time = perf_counter() - started
    RECORDS.append(record)
    row = {'trajectory_index': index, 'wall_time_s': wall_time, **trajectory_summary(record)}
    SUMMARY_ROWS.append(row)
    print(f"  {record.termination_reason}; elapsed={row['elapsed_s'] * 1e3:.3f} ms; wall={wall_time:.2f} s")

    if SIMULATION['save_results']:
        stem = trajectory_root / f'trajectory_{index:04d}'
        save_trajectory(
            record, MODEL, BEAMS, stem,
            metadata={
                'physics_parameters': PHYSICS,
                'simulation_parameters': SIMULATION,
                'rate_config': asdict(RATE_CONFIG),
                'apparatus_config': asdict(APPARATUS),
                'coil_config': asdict(COIL),
                'launch': LAUNCH_METADATA[index],
                'trajectory_index': index,
            },
        )

SUMMARY = pd.DataFrame(SUMMARY_ROWS)
print(f'Ensemble wall time: {perf_counter() - ensemble_started:.2f} s')
display(SUMMARY)

if SIMULATION['save_results']:
    trajectory_root.mkdir(parents=True, exist_ok=True)
    SUMMARY.to_csv(trajectory_root / 'ensemble_summary.csv', index=False)
    manifest = {
        'schema': 'pmot.mot_multilevel.configured-trajectory-notebook.v1',
        'model': 'Section-12 24-state deterministic population-rate equation',
        'physics_parameters': PHYSICS,
        'simulation_parameters': SIMULATION,
        'rate_config': asdict(RATE_CONFIG),
        'apparatus_config': asdict(APPARATUS),
        'coil_config': asdict(COIL),
        'launches': LAUNCH_METADATA,
    }
    (trajectory_root / 'campaign_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
    print(f'Saved trajectory data to {trajectory_root}')

## 8. Three-dimensional trajectory and time diagnostics

Set `selected_trajectory_index` in the simulation dictionary and rerun this cell to inspect another ensemble member. The 3D plot includes the initial point, complete path, final point, trap center, true-scale beam envelope, and time coloring. The second figure shows position, velocity, beam-resolved net stimulated rates, optical force, spontaneous scattering, and manifold populations.

In [ ]:
if not RECORDS:
    raise RuntimeError('Run the trajectory cell first')
selected_index = int(SIMULATION['selected_trajectory_index'])
if not 0 <= selected_index < len(RECORDS):
    raise IndexError(f'selected_trajectory_index must be between 0 and {len(RECORDS) - 1}')
selected_record = RECORDS[selected_index]

trajectory_figure_path = None
diagnostics_figure_path = None
if SIMULATION['save_results']:
    figure_root.mkdir(parents=True, exist_ok=True)
    trajectory_figure_path = figure_root / f'trajectory_{selected_index:04d}_3d.png'
    diagnostics_figure_path = figure_root / f'trajectory_{selected_index:04d}_diagnostics.png'

TRAJECTORY_FIGURE = plot_trajectory_3d(
    selected_record, BEAMS, trajectory_figure_path,
    title=f'Configured multilevel MOT trajectory {selected_index}',
)
DIAGNOSTICS_FIGURE = plot_time_diagnostics(
    selected_record, MODEL, BEAMS, diagnostics_figure_path,
)
plt.show()

## 9. Trajectory animation

This creates an inline animation for the selected trajectory. `animation_max_frames` is a rendering and memory cap: the animation automatically uses one coarse timestep stride across the complete stored trajectory and always includes its first and final samples. The inline display uses compressed HTML5 video when available instead of embedding every frame as a separate image. Enable `save_animation_gif` before running this cell if a GIF should also be written to the figures directory. The full coarse-grained path is shown faintly; the dark trace and red marker advance through simulated time.

In [ ]:
if not RECORDS:
    raise RuntimeError('Run the trajectory cell first')
selected_index = int(SIMULATION['selected_trajectory_index'])
selected_record = RECORDS[selected_index]
gif_path = None
if SIMULATION['save_animation_gif']:
    figure_root.mkdir(parents=True, exist_ok=True)
    gif_path = figure_root / f'trajectory_{selected_index:04d}.gif'

requested_frames = int(SIMULATION['animation_max_frames'])
available_samples = len(selected_record.times_s)
print(
    f'Animating up to {min(requested_frames, available_samples)} evenly spaced frames '
    f'across t={1e3*selected_record.times_s[0]:.3f} to '
    f'{1e3*selected_record.times_s[-1]:.3f} ms.'
)
TRAJECTORY_ANIMATION = create_trajectory_animation(
    selected_record,
    BEAMS,
    gif_path,
    max_frames=int(SIMULATION['animation_max_frames']),
    fps=int(SIMULATION['animation_fps']),
)
try:
    inline_animation = TRAJECTORY_ANIMATION.to_html5_video()
except Exception as error:
    inline_animation = ''
    print(f'Compressed video encoding unavailable ({error}); using a low-memory HTML fallback.')
if '<video' not in inline_animation:
    fallback_frames = min(requested_frames, 40)
    plt.close(TRAJECTORY_ANIMATION._fig)
    TRAJECTORY_ANIMATION = create_trajectory_animation(
        selected_record, BEAMS, None, max_frames=fallback_frames,
        fps=int(SIMULATION['animation_fps']),
    )
    inline_animation = TRAJECTORY_ANIMATION.to_jshtml()
display(HTML(inline_animation))
if gif_path is not None:
    print(f'Saved animation to {gif_path}')
plt.close(TRAJECTORY_ANIMATION._fig)

## Interpretation boundary

These trajectories use the replacement physical population-rate kernel and are suitable for configuration diagnostics, force/trajectory inspection, and convergence work. They do **not** include recoil diffusion and therefore cannot establish a temperature. A visually bounded trajectory is also not, by itself, a converged capture classification. Production capture/loading work must use the repository's fail-closed capture interfaces, timestep and duration audits, independent direction-disc statistics, and representative event-engine comparisons.